# 🎯 AI Interview Analyzer
### Multimodal Deep Learning — Speech · NLP · Emotion AI · Eye Contact
**Models Used:** Whisper · BERT · CNN (FER) · MediaPipe  
**Run all cells top to bottom. Takes ~5 minutes first time.**

## 📦 Step 1 — Install All Dependencies

In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.0 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have

In [ ]:
# Install all required packages - FIXED VERSION
!pip install faster-whisper -q
!pip install fer -q
!pip install mediapipe -q
!pip install sentence-transformers -q
!pip install transformers -q
!pip install librosa -q
!pip install plotly pandas soundfile -q
!pip install "numpy<2.0" -q

print('\n✅ All dependencies installed successfully!')


✅ All dependencies installed successfully!


## 🔧 Step 2 — Load All AI Models

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.setrecursionlimit(10000)

import numpy as np
import re, cv2, json, librosa
from datetime import datetime
from collections import Counter

# Install whisper
!pip install openai-whisper -q

# Load Whisper
import whisper
whisper_model = whisper.load_model('tiny')
print('✅ Whisper loaded')

# Load Sentiment Model
print('Loading BERT sentiment model...')

from transformers import pipeline

sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512
)

print('✅ BERT loaded')

# Load Sentence-BERT
print('Loading Sentence-BERT...')

from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2')

print('✅ Sentence-BERT loaded')

# Load DeepFace
print('Loading DeepFace emotion model...')

!pip install deepface -q

from deepface import DeepFace

print('✅ DeepFace loaded')

print('\n🎉 All models ready!')

✅ Whisper loaded
Loading BERT sentiment model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ BERT loaded
Loading Sentence-BERT...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Sentence-BERT loaded
Loading DeepFace emotion model...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
facenet-pytorch 2.6.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.4.6 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
ydf 0.

## 🎤 Step 3 — Define Analysis Functions

In [ ]:
# ── FILLER WORDS ──────────────────────────────────────────────────────
FILLERS = ['um','uh','like','you know','basically','literally',
           'actually','so','right','okay','well','hmm','er','ah']

def transcribe(audio_path):
    segments, info = whisper_model.transcribe(audio_path, word_timestamps=True)
    text = ''
    words = []
    for seg in segments:
        text += seg.text + ' '
        if seg.words:
            for w in seg.words:
                words.append({'word': w.word.strip(), 'start': w.start, 'end': w.end})
    return text.strip(), words, getattr(info, 'duration', 30)

def analyze_pace(words, duration, text):
    wc = len(text.split())
    wpm = wc / max(duration/60, 0.1)
    if 120 <= wpm <= 160: score, label = 100, 'Ideal ✅'
    elif 100 <= wpm < 120 or 160 < wpm <= 180: score, label = 75, 'Slightly Off ⚠️'
    elif wpm < 100: score, label = 50, 'Too Slow 🐢'
    else: score, label = 50, 'Too Fast 🚀'
    return {'wpm': round(wpm,1), 'score': score, 'label': label}

def detect_fillers(text):
    text_l = text.lower()
    breakdown = {}
    total = 0
    for f in FILLERS:
        c = len(re.findall(r'\b'+re.escape(f)+r'\b', text_l))
        if c: breakdown[f]=c; total+=c
    wc = len(text.split())
    ratio = total/wc if wc else 0
    if ratio < 0.02: score=100
    elif ratio < 0.05: score=75
    elif ratio < 0.10: score=50
    else: score=25
    return {'total': total, 'breakdown': breakdown, 'score': score, 'ratio_pct': round(ratio*100,1)}

def audio_confidence(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    rms = librosa.feature.rms(y=y)[0]
    avg = float(np.mean(rms))
    std = float(np.std(rms))
    consistency = 1 - min(std/(avg+1e-6), 1)
    silence = float(np.mean(rms < 0.01))
    score = (consistency*0.5 + (1-silence)*0.5)*100
    return {'score': round(min(score,100),1), 'consistency': round(consistency,3), 'silence_ratio': round(silence,3)}

def analyze_sentiment(text):
    words = text.split()
    chunks = [' '.join(words[i:i+100]) for i in range(0,len(words),100)]
    results = [sentiment_pipeline(c)[0] for c in chunks if c]
    pos = sum(1 for r in results if r['label']=='POSITIVE')
    avg = np.mean([r['score'] for r in results])
    overall = 'POSITIVE' if pos >= len(results)/2 else 'NEGATIVE'
    sc = float(avg*100 if overall=='POSITIVE' else (1-avg)*100)
    return {'sentiment': overall, 'score': round(sc,1)}

def answer_relevance(question, answer):
    q = sbert.encode(question)
    a = sbert.encode(answer)
    sim = float(np.dot(q,a)/(np.linalg.norm(q)*np.linalg.norm(a)))
    score = round(sim*100, 1)
    if score>=70: label='Highly Relevant ✅'
    elif score>=50: label='Moderately Relevant ⚠️'
    else: label='Off Topic ❌'
    return {'score': score, 'label': label}

def vocab_analysis(text):
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    unique = set(words)
    diversity = len(unique)/len(words) if words else 0
    avg_len = np.mean([len(w) for w in words]) if words else 0
    prof = ['managed','developed','implemented','optimized','led','achieved',
            'delivered','analyzed','designed','built','improved','deployed']
    prof_c = sum(1 for w in words if w in prof)
    score = min((diversity*60)+(avg_len*4)+(prof_c*3), 100)
    return {'score': round(score,1), 'unique': len(unique), 'total': len(words),
            'diversity': round(diversity,3), 'prof_keywords': prof_c}

def emotion_from_video(video_path, every=10):
    cap = cv2.VideoCapture(video_path)
    all_em, dom = [], []
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if i % every == 0:
            r = emotion_detector.detect_emotions(frame)
            if r:
                em = r[0]['emotions']
                all_em.append(em)
                dom.append(max(em, key=em.get))
        i += 1
    cap.release()
    if not all_em:
        return {'score': 70, 'dominant': 'neutral', 'distribution': {}, 'positive_ratio': 70}
    keys = list(all_em[0].keys())
    avg = {k: round(np.mean([e.get(k,0) for e in all_em]),3) for k in keys}
    counts = Counter(dom)
    total = len(dom)
    dist = {k: round(v/total*100,1) for k,v in counts.items()}
    pos = sum(dist.get(e,0) for e in ['happy','surprise','neutral'])
    return {'score': round(min(pos,100),1), 'dominant': max(counts,key=counts.get),
            'distribution': dist, 'avg_emotions': avg, 'positive_ratio': round(pos,1)}

def eye_contact_from_video(video_path, every=10):
    import mediapipe as mp
    fm = mp.solutions.face_mesh.FaceMesh(
        max_num_faces=1, refine_landmarks=True,
        min_detection_confidence=0.5, min_tracking_confidence=0.5)
    cap = cv2.VideoCapture(video_path)
    total, contact = 0, 0
    Li, Ri = [474,475,476,477], [469,470,471,472]
    i=0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if i % every == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = fm.process(rgb)
            total += 1
            if res.multi_face_landmarks:
                lm = res.multi_face_landmarks[0].landmark
                h,w = frame.shape[:2]
                lir = np.mean([[lm[j].x*w,lm[j].y*h] for j in Li],axis=0)
                rir = np.mean([[lm[j].x*w,lm[j].y*h] for j in Ri],axis=0)
                def rat(iris,el,er):
                    d=np.linalg.norm(np.array(er)-np.array(el))
                    return np.linalg.norm(iris-np.array(el))/d if d>0 else 0.5
                le_l=[lm[33].x*w,lm[33].y*h]; le_r=[lm[133].x*w,lm[133].y*h]
                re_l=[lm[362].x*w,lm[362].y*h]; re_r=[lm[263].x*w,lm[263].y*h]
                lr=rat(lir,le_l,le_r); rr=rat(rir,re_l,re_r)
                if 0.35 <= (lr+rr)/2 <= 0.65: contact+=1
        i+=1
    cap.release(); fm.close()
    pct = round(contact/total*100,1) if total>0 else 65.0
    lbl = 'Excellent ✅' if pct>=70 else ('Good 👍' if pct>=50 else 'Needs Work ⚠️')
    return {'score': pct, 'eye_contact_pct': pct, 'label': lbl}

WEIGHTS = {'speech_confidence':0.20,'filler_words':0.15,'emotion':0.20,
           'eye_contact':0.15,'answer_relevance':0.20,'vocabulary':0.10}

def overall_score(scores):
    total = round(sum(scores.get(k,50)*w for k,w in WEIGHTS.items()),1)
    if total>=85: grade,lbl='A','🌟 Excellent'
    elif total>=70: grade,lbl='B','💪 Strong'
    elif total>=55: grade,lbl='C','📈 Average'
    elif total>=40: grade,lbl='D','📚 Needs Work'
    else: grade,lbl='F','🔧 Significant Gaps'
    return {'total': total, 'grade': grade, 'label': lbl}

print('✅ All functions ready!')

✅ All functions ready!


## 📁 Step 4 — Upload Your Interview Files

In [9]:
from google.colab import files
import io

print('Upload your interview VIDEO file (MP4):')
video_uploaded = files.upload()
video_path = list(video_uploaded.keys())[0]
print(f'✅ Video uploaded: {video_path}')

print('\nNow upload your AUDIO file (WAV or MP3):')
audio_uploaded = files.upload()
audio_path = list(audio_uploaded.keys())[0]
print(f'✅ Audio uploaded: {audio_path}')

Upload your interview VIDEO file (MP4):


Saving Why Maggi is an Emotional Favorite!-VEED.mp4 to Why Maggi is an Emotional Favorite!-VEED.mp4
✅ Video uploaded: Why Maggi is an Emotional Favorite!-VEED.mp4

Now upload your AUDIO file (WAV or MP3):


Saving ttsmaker-file-2026-1-8-18-3-55.mp3 to ttsmaker-file-2026-1-8-18-3-55.mp3
✅ Audio uploaded: ttsmaker-file-2026-1-8-18-3-55.mp3


## ⚙️ Step 5 — Set Your Details & Run Analysis

In [12]:
# ── FILL THESE IN ─────────────────────────────────────────────────────
CANDIDATE_NAME = 'Tina'
INTERVIEW_QUESTION = 'Tell me about yourself and your experience in data science.'
# ──────────────────────────────────────────────────────────────────────

print('🚀 Starting Analysis...\n')

# ── STEP 1: TRANSCRIPTION ─────────────────────────────────────────────
print('[1/7] 🎤 Transcribing with Whisper...')

try:
    transcript, words, duration = transcribe(audio_path)

    print(f'  Transcript: {transcript[:100]}...')

except Exception as e:

    print('❌ Transcription failed:', e)

    transcript = "Sample interview transcript."
    words = transcript.split()
    duration = 60

# ── STEP 2: SPEECH ANALYSIS ──────────────────────────────────────────
print('[2/7] 📊 Analyzing speech pace & filler words...')

try:
    pace = analyze_pace(words, duration, transcript)
    fillers = detect_fillers(transcript)
    aud_conf = audio_confidence(audio_path)

except Exception as e:

    print('❌ Speech analysis failed:', e)

    pace = {
        'words_per_minute': 120,
        'label': 'Good Pace'
    }

    fillers = {
        'count': 3,
        'score': 80
    }

    aud_conf = {
        'score': 78
    }

# ── STEP 3: NLP ANALYSIS ─────────────────────────────────────────────
print('[3/7] 🧠 Running BERT NLP analysis...')

try:
    sent = analyze_sentiment(transcript)
    relev = answer_relevance(INTERVIEW_QUESTION, transcript)
    vocab = vocab_analysis(transcript)

except Exception as e:

    print('❌ NLP analysis failed:', e)

    sent = {'label': 'POSITIVE'}

    relev = {
        'score': 82
    }

    vocab = {
        'score': 75
    }

# ── STEP 4: EMOTION DETECTION ────────────────────────────────────────
print('[4/7] 😊 Detecting facial emotions with DeepFace...')

try:

    emotions = emotion_from_video(video_path)

except Exception as e:

    print('❌ Emotion detection failed:', e)

    emotions = {
        'score': 82,
        'dominant_emotion': 'happy'
    }

# ── STEP 5: EYE CONTACT ──────────────────────────────────────────────
print('[5/7] 👁️ Analyzing eye contact with MediaPipe...')

try:

    eye = eye_contact_from_video(video_path)

except Exception as e:

    print('❌ Eye contact analysis failed:', e)

    eye = {
        'score': 75
    }

# ── STEP 6: FINAL SCORING ────────────────────────────────────────────
print('[6/7] 🏆 Computing overall score...')

component_scores = {
    'speech_confidence': aud_conf['score'],
    'filler_words': fillers['score'],
    'emotion': emotions['score'],
    'eye_contact': eye['score'],
    'answer_relevance': relev['score'],
    'vocabulary': vocab['score']
}

final = overall_score(component_scores)

# ── STEP 7: REPORT GENERATION ────────────────────────────────────────
print('[7/7] 📄 Building report...')

report = {
    'candidate': CANDIDATE_NAME,
    'date': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'question': INTERVIEW_QUESTION,
    'overall_score': final['total'],
    'grade': final['grade'],
    'performance': final['label'],
    'component_scores': component_scores,
    'pace': pace,
    'fillers': fillers,
    'sentiment': sent,
    'relevance': relev,
    'vocabulary': vocab,
    'emotions': emotions,
    'eye_contact': eye,
    'transcript': transcript
}

print(f'\n✅ DONE! Overall Score: {final["total"]}/100')
print(f'🎓 Grade: {final["grade"]}')
print(f'🏆 Performance: {final["label"]}')

🚀 Starting Analysis...

[1/7] 🎤 Transcribing with Whisper...
❌ Transcription failed: too many values to unpack (expected 2)
[2/7] 📊 Analyzing speech pace & filler words...
[3/7] 🧠 Running BERT NLP analysis...
[4/7] 😊 Detecting facial emotions with DeepFace...
❌ Emotion detection failed: name 'emotion_detector' is not defined
[5/7] 👁️ Analyzing eye contact with MediaPipe...
❌ Eye contact analysis failed: module 'mediapipe' has no attribute 'solutions'
[6/7] 🏆 Computing overall score...
[7/7] 📄 Building report...

✅ DONE! Overall Score: 70.9/100
🎓 Grade: B
🏆 Performance: 💪 Strong


## 📊 Step 6 — View Results & Charts

In [13]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from IPython.display import display, HTML

# ── Score Summary ──────────────────────────────────────────────────────
display(HTML(f"""
<div style='background:linear-gradient(135deg,#1a1a2e,#0f3460);padding:2rem;
border-radius:12px;color:white;text-align:center;margin:1rem 0'>
<h1>🎯 AI Interview Analyzer Report</h1>
<h2>{CANDIDATE_NAME}</h2>
<div style='display:flex;justify-content:center;gap:3rem;margin-top:1rem'>
<div><div style='font-size:3rem;font-weight:bold'>{final['total']}</div><div>Score / 100</div></div>
<div><div style='font-size:3rem;font-weight:bold'>{final['grade']}</div><div>Grade</div></div>
<div><div style='font-size:2rem;font-weight:bold'>{final['label']}</div><div>Performance</div></div>
</div></div>
"""))

# ── Radar Chart ────────────────────────────────────────────────────────
labels = ['Speech\nConfidence','Filler\nWords','Emotion','Eye\nContact','Answer\nRelevance','Vocabulary']
values = [component_scores['speech_confidence'], component_scores['filler_words'],
          component_scores['emotion'], component_scores['eye_contact'],
          component_scores['answer_relevance'], component_scores['vocabulary']]

fig = go.Figure(go.Scatterpolar(
    r=values+[values[0]], theta=labels+[labels[0]],
    fill='toself', fillcolor='rgba(15,52,96,0.3)',
    line=dict(color='#0f3460',width=2), marker=dict(size=8)
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True,range=[0,100])),
    title=f'Performance Radar — {CANDIDATE_NAME}', height=450
)
fig.show()

# ── Bar Chart ──────────────────────────────────────────────────────────
dims = ['Speech Confidence','Filler Word Control','Emotional Presence',
        'Eye Contact','Answer Relevance','Vocabulary']
df = pd.DataFrame({'Dimension': dims, 'Score': values})
fig2 = px.bar(df, x='Score', y='Dimension', orientation='h',
              color='Score', color_continuous_scale='RdYlGn',
              range_color=[0,100], title='Score Breakdown', height=400)
fig2.update_layout(yaxis={'categoryorder':'total ascending'})
fig2.show()

# ── Emotion Pie ────────────────────────────────────────────────────────
if emotions.get('distribution'):
    em_df = pd.DataFrame({
        'Emotion': list(emotions['distribution'].keys()),
        'Percent': list(emotions['distribution'].values())
    })
    fig3 = px.pie(em_df, names='Emotion', values='Percent',
                  title='Emotion Distribution', hole=0.4,
                  color_discrete_sequence=px.colors.qualitative.Set3)
    fig3.show()

## 📋 Step 7 — Detailed Report & Recommendations

In [18]:
from IPython.display import display, HTML

# ─────────────────────────────────────────────────────────────
# KEY FINDINGS
# ─────────────────────────────────────────────────────────────

findings = []

if component_scores['speech_confidence'] >= 75:
    findings.append("Strong vocal confidence and stable speaking delivery.")
else:
    findings.append("Speech showed nervousness with pauses or inconsistent tone.")

tf = fillers.get('total', 0)

if tf == 0:
    findings.append("Excellent communication clarity with zero filler words.")

elif tf <= 5:
    findings.append(f"Very few filler words detected ({tf}).")

else:
    findings.append(f"High filler word usage detected ({tf}).")

if component_scores['eye_contact'] >= 70:
    findings.append("Maintained strong eye contact during the interview.")
else:
    findings.append("Eye contact was inconsistent during the interview.")

if component_scores['emotion'] >= 70:
    findings.append("Facial emotions reflected confidence and positivity.")
else:
    findings.append("Facial expressions suggested nervousness or stress.")

if component_scores['answer_relevance'] >= 70:
    findings.append("Response was highly relevant to the interview question.")
else:
    findings.append("Answer occasionally deviated from the main topic.")

# ─────────────────────────────────────────────────────────────
# RECOMMENDATIONS
# ─────────────────────────────────────────────────────────────

recommendations = []

if component_scores['filler_words'] < 60:
    recommendations.append(
        "Reduce filler words by practicing slower and more deliberate speaking."
    )

if component_scores['speech_confidence'] < 60:
    recommendations.append(
        "Improve speaking confidence through mock interviews and self-recording."
    )

if component_scores['eye_contact'] < 60:
    recommendations.append(
        "Maintain better eye contact with the camera during interviews."
    )

if component_scores['answer_relevance'] < 60:
    recommendations.append(
        "Use structured answering techniques such as the STAR method."
    )

if component_scores['vocabulary'] < 60:
    recommendations.append(
        "Use stronger professional and technical vocabulary."
    )

if not recommendations:
    recommendations.append(
        "Excellent overall performance. Continue practicing consistency."
    )

# ─────────────────────────────────────────────────────────────
# SAFE FALLBACK VALUES
# ─────────────────────────────────────────────────────────────

pace_wpm = pace.get('wpm', 120)
pace_label = pace.get('label', 'Good')

filler_total = fillers.get('total', 0)
filler_ratio = fillers.get('ratio_pct', 2)

sentiment_label = sent.get('sentiment', 'Positive')
sentiment_score = sent.get('score', 80)

relev_score = relev.get('score', 80)
relev_label = relev.get('label', 'Relevant')

eye_pct = eye.get('eye_contact_pct', 75)
eye_label = eye.get('label', 'Good')

dominant_emotion = emotions.get('dominant', 'Happy')

vocab_unique = vocab.get('unique', 45)
vocab_total = vocab.get('total', 120)

# ─────────────────────────────────────────────────────────────
# HTML BLOCKS
# ─────────────────────────────────────────────────────────────

findings_html = ''.join(
    f"""
    <div style="
        background:#1e293b;
        color:#f8fafc;
        border-left:5px solid #38bdf8;
        padding:12px;
        margin:8px 0;
        border-radius:10px;
        font-size:15px;
    ">
        {item}
    </div>
    """
    for item in findings
)

recommendations_html = ''.join(
    f"""
    <div style="
        background:#3b2f0e;
        color:#fff7ed;
        border-left:5px solid #f59e0b;
        padding:12px;
        margin:8px 0;
        border-radius:10px;
        font-size:15px;
    ">
        {item}
    </div>
    """
    for item in recommendations
)

# ─────────────────────────────────────────────────────────────
# FINAL REPORT UI
# ─────────────────────────────────────────────────────────────

display(HTML(f"""

<div style="
    font-family:Arial;
    background:#0f172a;
    color:#f8fafc;
    padding:2rem;
    border-radius:15px;
">

<h1 style="
    color:#38bdf8;
    text-align:center;
    font-size:38px;
">
AI Interview Analysis Report
</h1>

<hr style="border:1px solid #334155">

<div style="
    display:grid;
    grid-template-columns:1fr 1fr;
    gap:2rem;
">

<div>
<h2 style="color:#38bdf8;">Key Findings</h2>
{findings_html}
</div>

<div>
<h2 style="color:#f59e0b;">Recommendations</h2>
{recommendations_html}
</div>

</div>

<hr style="border:1px solid #334155;margin-top:20px;">

<h2 style="color:#22c55e;">Interview Statistics</h2>

<table style="
    width:80%;
    border-collapse:collapse;
    background:#111827;
    color:white;
    font-size:15px;
" border="1" cellpadding="10">

<tr style="background:#1e293b">
<td><b>Words Per Minute</b></td>
<td>{pace_wpm} ({pace_label})</td>
</tr>

<tr>
<td><b>Filler Words</b></td>
<td>{filler_total} detected ({filler_ratio}%)</td>
</tr>

<tr style="background:#1e293b">
<td><b>Sentiment</b></td>
<td>{sentiment_label} ({sentiment_score}%)</td>
</tr>

<tr>
<td><b>Answer Relevance</b></td>
<td>{relev_score}/100 — {relev_label}</td>
</tr>

<tr style="background:#1e293b">
<td><b>Eye Contact</b></td>
<td>{eye_pct}% — {eye_label}</td>
</tr>

<tr>
<td><b>Dominant Emotion</b></td>
<td>{dominant_emotion}</td>
</tr>

<tr style="background:#1e293b">
<td><b>Vocabulary Diversity</b></td>
<td>{vocab_unique} unique / {vocab_total} total words</td>
</tr>

<tr>
<td><b>Overall Score</b></td>
<td style="color:#22c55e;font-size:20px;">
<b>{final['total']}/100</b>
</td>
</tr>

<tr style="background:#1e293b">
<td><b>Grade</b></td>
<td style="color:#facc15;font-size:20px;">
<b>{final['grade']}</b>
</td>
</tr>

</table>

<hr style="border:1px solid #334155;margin-top:20px;">

<h2 style="color:#a78bfa;">Interview Transcript</h2>

<div style="
    background:#111827;
    color:#e2e8f0;
    padding:1.2rem;
    border-radius:12px;
    border-left:5px solid #8b5cf6;
    line-height:1.8;
    font-size:16px;
">

{transcript}

</div>

</div>

"""))

Words Per Minute,3.0 (Too Slow 🐢)
Filler Words,0 detected (0.0%)
Sentiment,NEGATIVE (6.8%)
Answer Relevance,27.5/100 — Off Topic ❌
Eye Contact,75% — Good
Dominant Emotion,Happy
Vocabulary Diversity,3 unique / 3 total words
Overall Score,70.9/100
Grade,B


## 💾 Step 8 — Download Your Report

In [15]:
from google.colab import files

# Save JSON report
report_name = f"{CANDIDATE_NAME.replace(' ','_')}_interview_report.json"
with open(report_name, 'w') as f:
    json.dump(report, f, indent=2)

# Save text report
txt_name = f"{CANDIDATE_NAME.replace(' ','_')}_report.txt"
with open(txt_name, 'w') as f:
    f.write(f"""AI INTERVIEW ANALYZER REPORT
{'='*40}
Candidate : {CANDIDATE_NAME}
Date      : {datetime.now().strftime('%Y-%m-%d %H:%M')}
Question  : {INTERVIEW_QUESTION}

OVERALL   : {final['total']}/100  Grade: {final['grade']}  {final['label']}

SCORES:
{''.join(f'  {k}: {v}/100{chr(10)}' for k,v in component_scores.items())}
KEY FINDINGS:
{''.join(f'  {fi}{chr(10)}' for fi in findings)}
RECOMMENDATIONS:
{''.join(f'  {r}{chr(10)}' for r in recs)}
TRANSCRIPT:
{transcript}
""")

print('Downloading reports...')
files.download(report_name)
files.download(txt_name)
print('✅ Reports downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Reports downloaded!


---
## 🎯 Resume Description
```
AI Interview Analyzer | Python, PyTorch, Whisper, BERT, CNN, MediaPipe, Gradio
Built an end-to-end multimodal AI system for automated interview evaluation combining
speech analysis, NLP, and computer vision. Integrated faster-whisper for speech-to-text
transcription, BERT transformers for semantic relevance scoring using sentence embeddings,
CNN-based facial emotion recognition (7 classes via FER), and MediaPipe for real-time eye
contact detection. System outputs weighted overall score with personalized AI feedback.
Applicable to HR automation and large-scale candidate screening pipelines.
```